In [2]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import sys
import os

sys.path.append(os.path.abspath(".."))
import settings

def load_file(file_path):
    return pd.read_sas(file_path, format='xport')

def merge_files(file1, file2):
    return pd.merge(file1, file2[['SEQN','RIDAGEYR','RIAGENDR']], 
                    on='SEQN', how='left')

# Načítanie dát
bmx = load_file(settings.bmx_path)
demo = load_file(settings.demo_path)
data = merge_files(bmx, demo)

# Výber dospelých
adults = data[data['RIDAGEYR'] > 18]

# BMI pre mužov a ženy
menBMI = adults[adults['RIAGENDR'] == 1]['BMXBMI'].dropna()
womenBMI = adults[adults['RIAGENDR'] == 2]['BMXBMI'].dropna()

# Transformácie
transformationsMen = {
    "RAW": menBMI,
    "LOG": np.log(menBMI),
    "SQRT": np.sqrt(menBMI),
    "RECIP": 1 / menBMI
}

transformationsWomen = {
    "RAW": womenBMI,
    "LOG": np.log(womenBMI),
    "SQRT": np.sqrt(womenBMI),
    "RECIP": 1 / womenBMI
}

# Ukladanie výsledkov t-testu
results = []

# t-test
for key in transformationsMen.keys():
    men_sample = transformationsMen[key]
    women_sample = transformationsWomen[key]
    
    t_stat, p_val = ttest_ind(men_sample, women_sample, equal_var=False)  # Welch t-test
    results.append({
        "Transformácia": key,
        "p-hodnota": round(p_val, 8)  # desatinné čísla
    })

# Vytvorenie tabuľky
df_ttest = pd.DataFrame(results)
df_ttest.to_csv("bmi_ttest_transformations.csv", index=False)
display(df_ttest)

,Transformácia,p-hodnota
0,RAW,2.400000e-07
1,LOG,1.381800e-04
2,SQRT,6.040000e-06
3,RECIP,2.471605e-02
